# ⚽ eFootball AI Assistant - End-to-End Colab Training & Benchmark Pipeline

Pipeline tự động huấn luyện và đóng gói mô hình AI nhận diện thời gian thực (Real-time Vision AI) cho **eFootball Mobile**:
- **Mô hình 1: Field Detector** (Bóng, Cầu thủ, Thủ môn, Trọng tài, Con trỏ)
- **Mô hình 2: Pitch Calibration Keypoints** (32 điểm mốc sân bóng để tính ma trận phối cảnh 2D)
- **Mô hình 3: Minimap / Radar** (Nhận diện vị trí toàn bộ 22 cầu thủ trên bản đồ thu nhỏ)
- **Video Tracking Benchmark**: Kiểm thử tracking bóng trên video thực tế, đo FPS và Latency (ms)
- **Export Android Bundle**: Xuất mô hình ONNX / TFLite tối ưu độ trễ thấp để nạp vào Android APK.

## 1. Kết nối Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
import os, sys, shutil, subprocess
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Kiểm tra GPU
import torch
if not torch.cuda.is_available():
    raise SystemError('Vui lòng bật GPU trong Runtime -> Change runtime type -> T4 GPU!')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'✅ PyTorch Version: {torch.__version__}')

## 2. Thiết lập Môi trường & Clone Repository efootphy

In [ ]:
# 3. Clone Repository efootphy (nhánh upgrade/vision-fusion-qc)
WORKSPACE = Path('/content/efootphy')
if not WORKSPACE.exists():
    !git clone -b upgrade/vision-fusion-qc https://github.com/clonephy-blip/efootphy.git {WORKSPACE}
else:
    !git -C {WORKSPACE} pull

%cd {WORKSPACE}

# 4. Cài đặt các thư viện tự động
!pip install -q ultralytics onnx onnxruntime-gpu opencv-python-headless

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

import ultralytics
print(f'✅ Đã cài đặt thành công Ultralytics v{ultralytics.__version__}!')

## 3. Tự động Tìm kiếm và Kết nối Thư mục Dữ liệu train ai

In [ ]:
# Tự động tìm kiếm thư mục dữ liệu 'train ai' trong Google Drive
DATASET_LOCAL = Path('/content/datasets')
DATASET_LOCAL.mkdir(parents=True, exist_ok=True)
source_folder = None

# Danh sách các đường dẫn ưu tiên kiểm tra trong MyDrive
potential_paths = [
    Path('/content/drive/MyDrive/train ai'),
    Path('/content/drive/MyDrive/train_ai'),
    Path('/content/drive/MyDrive/efootball_detection'),
]

for p in potential_paths:
    if p.exists() and ((p / 'efootball_detection').exists() or p.name == 'efootball_detection'):
        source_folder = p if (p / 'efootball_detection').exists() else p.parent
        break

# Nếu chưa thấy ở vị trí gốc, quét đệ quy MyDrive (kể cả các shortcut / lối tắt)
if source_folder is None and os.path.exists('/content/drive/MyDrive'):
    for root, dirs, files in os.walk('/content/drive/MyDrive', followlinks=True):
        if 'efootball_detection' in dirs or os.path.basename(root) == 'train ai':
            candidate = Path(root)
            if (candidate / 'efootball_detection').exists():
                source_folder = candidate
                break
            elif candidate.name == 'train ai':
                source_folder = candidate
                break

if source_folder and (source_folder / 'efootball_detection').exists():
    print(f'✅ ĐÃ TÌM THẤY THƯ MỤC DỮ LIỆU: {source_folder}')
    print('Danh sách tệp/thư mục trong dataset:')
    for item in sorted(source_folder.iterdir()):
        print(f'  - {item.name}')
else:
    print('⚠️ CHƯA TÌM THẤY THƯ MỤC DỮ LIỆU TRONG MYDRIVE!')
    if os.path.exists('/content/drive/MyDrive'):
        print('Các thư mục hiện có trong MyDrive của bạn:', os.listdir('/content/drive/MyDrive'))
    print('\n👉 HƯỚNG DẪN 3 GIÂY ĐỂ KẾT NỐI DỮ LIỆU:')
    print('1. Chuyển sang Tab "train ai - Google Drive" đang mở sẵn trên trình duyệt Cốc Cốc của bạn.')
    print('2. Bấm vào chữ "train ai" ở đầu trang (cạnh biểu tượng thư mục) -> Chọn "Thêm lối tắt vào Drive" (Add shortcut to Drive).')
    print('3. Chọn "Drive của tôi" (My Drive) rồi bấm "Thêm" (Add).')
    print('4. Quay lại Colab và bấm ▶ Chạy tất cả là hoàn thành ngay!\n')
    raise FileNotFoundError('Vui lòng tạo lối tắt cho thư mục "train ai" vào "Drive của tôi" theo hướng dẫn ở trên!')

In [ ]:
# Chuẩn bị đường dẫn các dataset vào SSD của Colab để train tốc độ cao nhất
def get_dataset_path(folder_name):
    p = source_folder / folder_name
    p_local = DATASET_LOCAL / folder_name
    if p_local.exists() and (p_local / 'data.yaml').exists():
        return p_local
    if p.exists() and (p / 'data.yaml').exists():
        if not p_local.exists():
            print(f'Đang sao chép {folder_name} sang Colab SSD để huấn luyện tốc độ cao...')
            shutil.copytree(p, p_local)
        return p_local
    return None

field_dir = get_dataset_path('efootball_detection')
points_dir = get_dataset_path('efootball-analytics-points.v2i.yolov8')
minimap_dir = get_dataset_path('UI Mini Map')

print(f'Field Dataset: {field_dir}')
print(f'Points Dataset: {points_dir}')
print(f'Minimap Dataset: {minimap_dir}')

In [ ]:
import ast
import hashlib
import json
import os
import tempfile
import zipfile
from pathlib import Path

import onnx
import yaml

MINIMAP_CLASSES = ('ball', 'controlled', 'dot')
MAX_MODEL_BYTES = 40 * 1024 * 1024


def ordered_names(names):
    if isinstance(names, (list, tuple)):
        result = list(names)
    elif isinstance(names, dict):
        # Do not silently reorder class IDs or accept missing/duplicate IDs.
        if all(type(k) is int for k in names):
            if set(names) != set(range(len(names))):
                raise ValueError('Class IDs phải liên tục từ 0.')
            result = [names[i] for i in range(len(names))]
        elif all(type(k) is str for k in names):
            if set(names) != {str(i) for i in range(len(names))}:
                raise ValueError('Class IDs phải liên tục từ 0.')
            result = [names[str(i)] for i in range(len(names))]
        else:
            raise ValueError('Class IDs không hợp lệ.')
    else:
        raise ValueError('Thiếu danh sách nhãn của model/dataset.')
    if any(not isinstance(name, str) or not name for name in result):
        raise ValueError('Tên nhãn không hợp lệ.')
    return result


def require_minimap_names(names):
    actual = ordered_names(names)
    if actual != list(MINIMAP_CLASSES):
        raise ValueError(
            f'APK cần class IDs 0=ball, 1=controlled, 2=dot; đang có {actual}. '
            'Không tự đổi tên nhãn hoặc dùng model field thay minimap. '
            'Nếu cần đổi ID, phải chuyển cả các tệp nhãn dataset rồi train lại.'
        )
    return actual


def check_minimap_dataset(data_yaml):
    data = yaml.safe_load(Path(data_yaml).read_text(encoding='utf-8'))
    if not isinstance(data, dict):
        raise ValueError('data.yaml không phải cấu hình dataset hợp lệ.')
    require_minimap_names(data.get('names'))
    if 'nc' in data and (type(data['nc']) is not int or data['nc'] != 3):
        raise ValueError('Dataset minimap phải có đúng nc=3.')


def _check_tensor(value, expected):
    if not value.type.HasField('tensor_type'):
        raise ValueError('APK chỉ hỗ trợ tensor ONNX.')
    tensor = value.type.tensor_type
    dims = [d.dim_value if d.HasField('dim_value') else None
            for d in tensor.shape.dim]
    if tensor.elem_type != onnx.TensorProto.FLOAT or dims != list(expected):
        raise ValueError(f'Tensor {value.name}: cần FLOAT32 {list(expected)}, có {dims}.')


def validate_minimap_bytes(model_bytes):
    if not model_bytes or len(model_bytes) > MAX_MODEL_BYTES:
        raise ValueError('Model rỗng hoặc vượt giới hạn 40 MiB của APK.')
    model = onnx.load_model_from_string(model_bytes)
    if any(onnx.external_data_helper.uses_external_data(t)
           for t in model.graph.initializer):
        raise ValueError('Gói APK cần một model.onnx tự chứa trọng số, không external data.')
    if len(model.graph.input) != 1 or len(model.graph.output) != 1:
        raise ValueError('Model minimap cần đúng một input và một output.')
    _check_tensor(model.graph.input[0], (1, 3, 320, 320))
    _check_tensor(model.graph.output[0], (1, 7, 2100))
    metadata = {item.key: item.value for item in model.metadata_props}
    if len(metadata) != len(model.metadata_props):
        raise ValueError('Metadata ONNX có khóa trùng.')
    if metadata.get('task') != 'detect':
        raise ValueError('Model cần metadata task=detect.')
    try:
        names = ast.literal_eval(metadata['names'])
    except (KeyError, ValueError, SyntaxError) as exc:
        raise ValueError('Không xác minh được thứ tự nhãn từ metadata ONNX.') from exc
    require_minimap_names(names)
    # Structural validation only; accuracy and device inference remain separate QC.
    onnx.checker.check_model(model, full_check=True)
    return {
        'schema_version': 1,
        'profile': 'minimap',
        'task': 'detect',
        'input_size': 320,
        'input_layout': 'NCHW',
        'output_layout': 'channels_first',
        'normalization': 'RGB/255',
        'classes': list(MINIMAP_CLASSES),
        'model_sha256': hashlib.sha256(model_bytes).hexdigest(),
    }


def build_minimap_bundle(model_path, output_zip):
    model_path, output_zip = Path(model_path), Path(output_zip)
    if not model_path.is_file():
        raise FileNotFoundError(f'Không có ONNX minimap vừa export: {model_path}')
    if model_path.resolve() == output_zip.resolve():
        raise ValueError('Đường dẫn ZIP phải khác tệp ONNX nguồn.')
    if model_path.stat().st_size > MAX_MODEL_BYTES:
        raise ValueError('Model vượt giới hạn 40 MiB của APK.')
    model_bytes = model_path.read_bytes()
    manifest = validate_minimap_bytes(model_bytes)
    manifest_bytes = json.dumps(manifest, ensure_ascii=False, indent=2).encode('utf-8')
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix='.minimap-', suffix='.zip', dir=output_zip.parent)
    os.close(fd)
    try:
        with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            archive.writestr('model.onnx', model_bytes)
            archive.writestr('manifest.json', manifest_bytes)
        with zipfile.ZipFile(temporary) as archive:
            if archive.namelist() != ['model.onnx', 'manifest.json'] or archive.testzip() is not None:
                raise ValueError('ZIP không đúng cấu trúc của APK.')
            saved_manifest = json.loads(archive.read('manifest.json'))
            if hashlib.sha256(archive.read('model.onnx')).hexdigest() != saved_manifest['model_sha256']:
                raise ValueError('SHA-256 của model trong ZIP không khớp.')
        os.replace(temporary, output_zip)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)
    return manifest


In [ ]:
# Kiểm tra nhãn minimap trước các lượt train tốn GPU.
if minimap_dir is not None:
    check_minimap_dataset(minimap_dir / 'data.yaml')
    print('Dataset minimap đúng thứ tự: ball, controlled, dot.')


## 4. Huấn luyện Mô hình 1: Field Detector (Bóng + Cầu thủ + Trọng tài)

Sử dụng kiến trúc **YOLO11n (Nano)** siêu nhẹ để tối ưu độ trễ (cần đo độ trễ trên điện thoại thực tế).
- 7 Class: `['ball', 'cursor', 'cursor_secondary', 'goalkeeper', 'player', 'powerbar', 'referee']`
- Kích thước ảnh: `320x320` cho ứng dụng mobile real-time.

In [ ]:
from ultralytics import YOLO

if field_dir and (field_dir / 'data.yaml').exists():
    field_data_yaml = field_dir / 'data.yaml'
    field_model = YOLO('yolo11n.pt')

    print('🚀 Bắt đầu huấn luyện mô hình Field (Bóng + Cầu thủ + Trọng tài)...')
    field_results = field_model.train(
        data=str(field_data_yaml),
        epochs=30,
        imgsz=320,          # 320px là kích thước input; FPS cần đo trên thiết bị
        batch=16,
        device=0,
        workers=4,
        patience=8,
        save=True,
        project='/content/runs/field',
        name='yolo11n_field_320',
        exist_ok=True
    )

    # Xuất sang định dạng ONNX
    field_onnx_path = field_model.export(format='onnx', imgsz=320, dynamic=False, opset=12)
    print(f'✅ Đã xuất mô hình Field sang ONNX: {field_onnx_path}')
else:
    print('❌ Không tìm thấy dataset Field.')

## 5. Huấn luyện Mô hình 2: Pitch Calibration Keypoints (32 mốc sân)

Mô hình Pose ước lượng 32 điểm nút trên mặt sân cỏ (góc sân, vòng tròn giữa sân, vạch 16m50). Tọa độ này kết hợp với module `Projection.kt` trong app để tính ma trận **Homography**, chuyển đổi mọi vị trí cầu thủ sang bản đồ 2D chiến thuật.

In [ ]:
if points_dir and (points_dir / 'data.yaml').exists():
    points_yaml = points_dir / 'data.yaml'
    pose_model = YOLO('yolo11n-pose.pt')

    print('🚀 Bắt đầu huấn luyện mô hình Sân bóng (32 Keypoints)...')
    pose_results = pose_model.train(
        data=str(points_yaml),
        epochs=30,
        imgsz=640,
        batch=8,
        device=0,
        workers=4,
        patience=8,
        save=True,
        project='/content/runs/pitch',
        name='yolo11n_pose_pitch',
        exist_ok=True
    )
    pose_onnx_path = pose_model.export(format='onnx', imgsz=640, dynamic=False, opset=12)
    print(f'✅ Đã xuất mô hình Pitch Pose sang ONNX: {pose_onnx_path}')
else:
    print('⚠️ Bỏ qua phần Pose do chưa có dataset.')

## 6. Huấn luyện Mô hình 3: UI Minimap Radar

Nhận diện 3 class trên minimap: `ball`, `controlled` (cầu thủ đang điều khiển), `dot` (các cầu thủ khác).

In [ ]:
from ultralytics import YOLO

# Clear the previous run's export reference before training; never reuse a guessed path.
minimap_onnx_path = None
minimap_bundle_path = None

if minimap_dir and (minimap_dir / 'data.yaml').exists():
    minimap_yaml = minimap_dir / 'data.yaml'
    check_minimap_dataset(minimap_yaml)
    minimap_model = YOLO('yolo11n.pt')

    print('Bắt đầu huấn luyện Minimap Radar...')
    minimap_results = minimap_model.train(
        data=str(minimap_yaml), epochs=30, imgsz=320, batch=16,
        device=0, workers=4, patience=8, save=True,
        project='/content/runs/minimap', name='yolo11n_minimap_320', exist_ok=True,
    )
    # Use best.pt from the actual training result, including any run-directory changes.
    minimap_best_pt = Path(minimap_results.save_dir) / 'weights' / 'best.pt'
    if not minimap_best_pt.is_file():
        raise FileNotFoundError(f'Train chưa tạo best.pt: {minimap_best_pt}')
    minimap_export_model = YOLO(str(minimap_best_pt))
    require_minimap_names(minimap_export_model.names)
    exported = Path(minimap_export_model.export(
        format='onnx', imgsz=320, batch=1, dynamic=False, opset=12,
        half=False, nms=False,
    )).resolve()
    # Assign only after the exported file passes the Android minimap contract.
    validate_minimap_bytes(exported.read_bytes())
    minimap_onnx_path = exported
    print(f'Đã kiểm tra ONNX minimap: {minimap_onnx_path}')
else:
    print('Không có dataset minimap; bước đóng gói sẽ dừng, không dùng model field thay thế.')


## 7. Video Tracking Benchmark & Kiểm định Độ trễ Realtime

Chạy thử nghiệm tracking trên video trận đấu từ Drive (`MATCH_01_PART_005.mp4`), đo độ trễ trên từng frame và xuất video preview kèm bounding box + quỹ đạo bóng.

In [ ]:
import cv2
import time
import numpy as np

# Tìm file video trận đấu từ source_folder
test_video = None
for vid_name in ['MATCH_01_PART_005.mp4', 'MATCH_01_PART_007.mp4', 'MATCH_01_PART_010.mp4']:
    candidate = source_folder / vid_name
    if candidate.exists():
        test_video = candidate
        break

best_field_model = Path('/content/runs/field/yolo11n_field_320/weights/best.pt')
if test_video and best_field_model.exists():
    print(f'🎬 Đang benchmark video: {test_video.name}')
    model = YOLO(str(best_field_model))
    cap = cv2.VideoCapture(str(test_video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    out_path = '/content/benchmark_replay_preview.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
    
    latencies = []
    ball_history = []
    frame_count = 0
    max_frames = 300  # Benchmark 300 frames (~10s)
    
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret: break
        
        t0 = time.perf_counter()
        # Nâng lên 640px và hạ conf=0.08 để bắt trọn quả bóng nhỏ (tránh bị nuốt pixel khi nén)
        results = model.predict(frame, imgsz=640, conf=0.08, verbose=False, device=0)
        t1 = time.perf_counter()
        
        latencies.append((t1 - t0) * 1000)
        frame_count += 1
        
        # Trích xuất và vẽ kết quả lên video
        res = results[0]
        for box in res.boxes:
            cls_id = int(box.cls[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_name = model.names[cls_id]
            
            if cls_name == 'ball':
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                ball_history.append((cx, cy))
                cv2.circle(frame, (cx, cy), 8, (0, 0, 255), -1)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            elif 'player' in cls_name or 'goalkeeper' in cls_name:
                color = (0, 255, 0) if cls_name == 'player' else (255, 165, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        # Vẽ vệt quỹ đạo bóng
        for i in range(1, len(ball_history[-20:])):
            cv2.line(frame, ball_history[-20:][i - 1], ball_history[-20:][i], (0, 255, 255), 2)
        
        # Vẽ thông số FPS & Latency
        curr_latency = latencies[-1]
        cv2.putText(frame, f'Latency: {curr_latency:.1f}ms | FPS: {1000/curr_latency:.1f}', (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        writer.write(frame)
    
    cap.release()
    writer.release()
    try:
        shutil.copy(out_path, '/content/drive/MyDrive/benchmark_replay_preview.mp4')
        print('💾 Đã tự động lưu video vào Google Drive của bạn (MyDrive/benchmark_replay_preview.mp4)')
    except Exception: pass
    
    avg_latency = np.mean(latencies)
    print(f'\n📊 KẾT QUẢ HIỆU NĂNG:')
    print(f'   - Độ trễ trung bình: {avg_latency:.2f} ms')
    print(f'   - Tốc độ xử lý: ~{1000/avg_latency:.1f} FPS')
    print(f'   - Video demo kết quả: {out_path}')
else:
    print('⚠️ Bỏ qua benchmark video do chưa có video hoặc model chưa train xong.')

## 8. Xuất gói Minimap cho Android

Chỉ đóng gói ONNX vừa xuất ở bước Minimap. Kiểm tra nhãn, FLOAT32, input/output,
graph, kích thước, cấu trúc ZIP và SHA-256; thiếu hoặc sai sẽ dừng.
Gói ZIP chỉ có `model.onnx` và `manifest.json` ở gốc, được lưu có dấu thời gian
trong `MyDrive/efootphy_exports/`. Field/Pose không được gắn nhãn thành Minimap.
Đây là kiểm tra tương thích cấu trúc, chưa xác nhận độ chính xác hoặc 60 FPS trên máy.


In [ ]:
from datetime import datetime, timezone
from google.colab import files
import shutil

# No fallback, no glob, no hardcoded best.onnx from an earlier run.
minimap_bundle_path = None
if globals().get('minimap_onnx_path') is None:
    raise RuntimeError(
        'Chưa có model minimap hợp lệ của lượt chạy này. '
        'Chạy bước train/export Minimap trước; không dùng ONNX field hoặc pose.'
    )

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
local_zip = Path('/content') / f'efootphy_minimap_bundle_{stamp}.zip'
manifest = build_minimap_bundle(minimap_onnx_path, local_zip)
minimap_bundle_path = local_zip

drive_root = Path('/content/drive/MyDrive')
if not drive_root.is_dir():
    raise RuntimeError(f'Chưa kết nối Drive. ZIP hợp lệ đã có tại {local_zip}')
drive_folder = drive_root / 'efootphy_exports'
drive_folder.mkdir(parents=True, exist_ok=True)
drive_zip = drive_folder / local_zip.name
shutil.copyfile(local_zip, drive_zip)
if hashlib.sha256(drive_zip.read_bytes()).digest() != hashlib.sha256(local_zip.read_bytes()).digest():
    raise IOError(f'Bản sao trên Drive không khớp; dùng ZIP cục bộ tại {local_zip}')

print(f'Đã xuất gói MINIMAP đúng cấu trúc APK: {drive_zip}')
print(f'Classes: {manifest["classes"]}')
print(f'Model SHA-256: {manifest["model_sha256"]}')
print('Kiểm tra cấu trúc thành công; độ chính xác và tốc độ cần kiểm tra trên thiết bị riêng.')
files.download(str(local_zip))
